# TRELLIS.2 — Seven-View vs Single-Image Demo (Colab A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TylerOlszewski/TRELLIS.2/blob/main/notebooks/TRELLIS2_MultiImage_Colab_A100.ipynb)

Generates two **textured 3D assets (GLB + turntable video)** from the same object for a
direct demo comparison:

1. A **seven-view run** using every 45° side/diagonal angle except straight-on front.
2. A **single-image run** using the held-out straight-on front image.

The seven-view demo uses `Trellis2ImageTo3DPipeline.run_multi_image()`; the single-image
demo uses the pipeline's native `run()` method. Camera poses are not required.

**Requirements**
- Colab **A100 GPU** and the **2025.10 runtime** (Runtime → Change runtime type).
  Pinning the runtime gives the notebook Python 3.12 + PyTorch 2.8 and keeps the compiled
  extension ABI reproducible as Colab's default image changes.
- A Hugging Face account with access to two **gated** models (see the auth cell below).
- Google Drive with a few GB free (used to cache compiled CUDA wheels between sessions).

**Timing**: the first run downloads a prebuilt FlashAttention wheel and compiles the other
CUDA extensions (typically ~15–40 min). Wheels are cached to Drive, so later sessions set
up in a few minutes. Model download and generation time are additional.

Run the cells top to bottom. Setup is idempotent: reconnecting or rerunning a cell reuses
every successfully cached wheel. The default seeds and resolution match between both runs
so the comparison changes only the image conditioning.

In [ ]:
#@title 1. Verify the pinned A100 runtime
!nvidia-smi
import sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab choose Runtime → Change runtime type → A100 GPU.')
name = torch.cuda.get_device_name(0)
runtime_ok = sys.version_info[:2] == (3, 12) and torch.__version__.split('+')[0].startswith('2.8.')
print(f"\nPython {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {name}")
if 'A100' not in name:
    raise RuntimeError(f'Expected an A100, but Colab assigned {name}. Change the hardware accelerator to A100.')
if not runtime_ok:
    raise RuntimeError(
        'Select Runtime → Change runtime type → Runtime Version 2025.10, then reconnect. '
        'This notebook intentionally pins Python 3.12 / PyTorch 2.8 for binary compatibility.'
    )
print('✓ compatible Colab A100 runtime')

In [ ]:
#@title 2. Mount Google Drive & set up caches
USE_DRIVE = True  #@param {type:"boolean"}
CACHE_MODELS_ON_DRIVE = False  #@param {type:"boolean"}

# Wheels (slow to build, small) always go to Drive when USE_DRIVE is on.
# Model checkpoints (~15 GB, fast to re-download) only go to Drive if you have the space.
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_ROOT = '/content/drive/MyDrive/TRELLIS2_cache'
else:
    CACHE_ROOT = '/content/TRELLIS2_cache'

WHEELS_DIR = f'{CACHE_ROOT}/wheels'
HF_HOME = f'{CACHE_ROOT}/hf_home' if (USE_DRIVE and CACHE_MODELS_ON_DRIVE) else '/content/hf_home'
os.makedirs(WHEELS_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
print('wheel cache :', WHEELS_DIR)
print('HF cache    :', HF_HOME)

In [ ]:
#@title 3. Prepare the CUDA build environment and wheel cache
# The pinned 2025.10 runtime has PyTorch/CUDA 12.6 and nvcc 12.5. PyTorch accepts a
# same-major minor-version difference when compiling extensions; a major mismatch is unsafe.
import os, re, shutil, subprocess, sys, pathlib, torch

def _versions():
    nvcc_path = shutil.which('nvcc')
    if nvcc_path is None:
        raise RuntimeError('nvcc is missing from this runtime; select Colab runtime 2025.10.')
    nvcc = subprocess.run([nvcc_path, '--version'], capture_output=True, text=True, check=True).stdout
    m = re.search(r'release (\d+)\.(\d+)', nvcc)
    if m is None or torch.version.cuda is None:
        raise RuntimeError('Could not determine the CUDA toolchain versions.')
    return nvcc_path, (int(m.group(1)), int(m.group(2))), tuple(int(x) for x in torch.version.cuda.split('.')[:2])

nvcc_path, (nv_maj, nv_min), (t_maj, t_min) = _versions()
print(f"system nvcc: {nv_maj}.{nv_min} | torch built for CUDA: {t_maj}.{t_min}")

if nv_maj != t_maj:
    raise RuntimeError(
        f'nvcc {nv_maj}.{nv_min} and torch CUDA {t_maj}.{t_min} have different major versions. '
        'Reconnect using Colab runtime 2025.10 instead of compiling an incompatible extension.'
    )
if nv_min != t_min:
    print('ℹ same CUDA major version; the 12.5/12.6 minor difference is expected on this runtime')

# CUDA_HOME must point at the toolkit that owns the active nvcc.
os.environ['CUDA_HOME'] = str(pathlib.Path(nvcc_path).resolve().parents[1])
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'  # compile only for the A100 architecture
os.environ['MAX_JOBS'] = str(min(8, os.cpu_count() or 2))

# Invalidate cached wheels when the torch/CUDA/python environment changed — wheels
# compiled against a different torch ABI crash at import time.
abi = int(torch.compiled_with_cxx11_abi())
env_tag = (f"torch{torch.__version__}-cu{torch.version.cuda}-nvcc{nv_maj}.{nv_min}-"
           f"py{sys.version_info.major}.{sys.version_info.minor}-abi{abi}-sm80")
tag_file = pathlib.Path(WHEELS_DIR) / 'env_tag.txt'
if tag_file.exists() and tag_file.read_text().strip() != env_tag:
    print(f"environment changed ({tag_file.read_text().strip()} → {env_tag}); clearing cached wheels…")
    for whl in pathlib.Path(WHEELS_DIR).glob('*.whl'):
        whl.unlink()
tag_file.write_text(env_tag)
print("wheel-cache tag:", env_tag)

In [ ]:
#@title 4. Clone repo & install Python dependencies
import os, subprocess, sys
REPO_URL = 'https://github.com/TylerOlszewski/TRELLIS.2.git'
REPO_REF = 'codex/colab-multi-image'  # Use 'main' after this PR is merged.
if not os.path.isdir('/content/TRELLIS.2'):
    !git clone --branch {REPO_REF} --single-branch {REPO_URL} /content/TRELLIS.2
else:
    !git -C /content/TRELLIS.2 fetch origin {REPO_REF}
    !git -C /content/TRELLIS.2 checkout --detach FETCH_HEAD
%cd /content/TRELLIS.2

deps = [
    'imageio', 'imageio-ffmpeg', 'tqdm', 'easydict', 'opencv-python-headless',
    'ninja', 'trimesh', 'kornia', 'timm', 'packaging', 'psutil', 'wheel',
    'plyfile', 'zstandard', 'matplotlib', 'huggingface_hub', 'pillow-heif',
    'transformers>=4.56.0,<5',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *deps], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8'],
               check=True)
print('✓ python deps installed')

In [ ]:
#@title 5. Hugging Face login (gated models)
# Two models used by the pipeline are GATED on Hugging Face — request access first
# (accept the license on each model page before running this cell):
#   • https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m   (image encoder)
#   • https://huggingface.co/briaai/RMBG-2.0                            (background removal)
# Then create a *read* token at https://huggingface.co/settings/tokens and add it to
# Colab Secrets (key icon in the left sidebar) under the name HF_TOKEN.
# This cell verifies both gated repositories before the expensive pipeline download.
import os
from huggingface_hub import get_token, hf_hub_download, login, whoami
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if token:
    os.environ['HF_TOKEN'] = token  # TRELLIS passes this directly to Transformers.
    login(token=token)
    print('✓ logged in with Colab secret HF_TOKEN')
else:
    print('No HF_TOKEN Colab secret found — falling back to interactive login:')
    login()
    token = get_token()

if not token:
    raise RuntimeError('Hugging Face login did not provide a token.')
os.environ['HF_TOKEN'] = token
print('Hugging Face account:', whoami(token=token)['name'])
for repo_id in ('facebook/dinov3-vitl16-pretrain-lvd1689m', 'briaai/RMBG-2.0'):
    hf_hub_download(repo_id, 'config.json', token=token)
    print('✓ gated-model access verified:', repo_id)

In [ ]:
#@title 6. Build / install CUDA extensions (cached on Drive after first run)
# FlashAttention uses an official prebuilt wheel. The five source-built wheels are pinned
# to immutable revisions and cached as each one succeeds. Rerunning this cell is safe.
import glob, os, shutil, subprocess, sys, torch, pathlib

EXT_ROOT = '/tmp/trellis2_extensions'
os.makedirs(EXT_ROOT, exist_ok=True)

# O-Voxel vendors Eigen as a header-only dependency, but the repository checkout
# intentionally leaves that directory empty. Install the distro headers once and
# expose them at the include path used by o-voxel/setup.py.
eigen_headers = pathlib.Path('/usr/include/eigen3/Eigen')
if not eigen_headers.exists():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libeigen3-dev'], check=True)
eigen_link = pathlib.Path('/content/TRELLIS.2/o-voxel/third_party/eigen/Eigen')
if not eigen_link.exists():
    eigen_link.symlink_to(eigen_headers, target_is_directory=True)
print('✓ Eigen headers ready for o-voxel')

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', *args], check=True)

def _clone(url, name, ref):
    dst = os.path.join(EXT_ROOT, name)
    if os.path.exists(dst) and not os.path.isdir(os.path.join(dst, '.git')):
        shutil.rmtree(dst)  # discard only an incomplete clone in the ephemeral /tmp tree
    if not os.path.exists(dst):
        subprocess.run(['git', 'clone', '--recursive', url, dst], check=True)
    subprocess.run(['git', '-C', dst, 'checkout', '--detach', ref], check=True)
    subprocess.run(['git', '-C', dst, 'submodule', 'update', '--init', '--recursive'], check=True)
    return dst

def _wheel(src):
    _pip('wheel', '--no-build-isolation', '--no-deps', '-w', WHEELS_DIR, src)

def _download(url):
    _pip('download', '--no-deps', '-d', WHEELS_DIR, url)

def ensure(import_name, wheel_prefix, build):
    try:
        __import__(import_name)
        print(f'✓ {import_name} already working')
        return
    except Exception:
        pass
    cached = sorted(glob.glob(f'{WHEELS_DIR}/{wheel_prefix}-*.whl'))
    if cached:
        print(f'→ installing {import_name} from cached wheel: {os.path.basename(cached[-1])}')
        _pip('install', '-q', '--force-reinstall', '--no-deps', cached[-1])
        try:
            __import__(import_name)
            print(f'✓ {import_name} (from cache)')
            return
        except Exception as e:
            print(f'  cached wheel unusable ({type(e).__name__}) — replacing it…')
            for path in cached:
                os.unlink(path)
    print(f'→ preparing {import_name} (source builds can take a while)…')
    build()
    built = sorted(glob.glob(f'{WHEELS_DIR}/{wheel_prefix}-*.whl'))
    assert built, f'build produced no wheel matching {wheel_prefix}-*.whl in {WHEELS_DIR}'
    _pip('install', '-q', '--force-reinstall', '--no-deps', built[-1])
    __import__(import_name)
    print(f'✓ {import_name} (installed and cached)')

ABI_LABEL = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
FLASH_WHEEL = (
    'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/'
    f'flash_attn-2.8.3+cu12torch2.8cxx11abi{ABI_LABEL}-cp312-cp312-linux_x86_64.whl'
)
ensure('flash_attn', 'flash_attn',
       lambda: _download(FLASH_WHEEL))
ensure('nvdiffrast', 'nvdiffrast',
       lambda: _wheel(_clone('https://github.com/NVlabs/nvdiffrast.git', 'nvdiffrast',
                                  '253ac4fcea7de5f396371124af597e6cc957bfae')))
ensure('nvdiffrec_render', 'nvdiffrec_render',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/nvdiffrec.git', 'nvdiffrec',
                                  'b296927cc7fd01c2ac1087c8065c4d7248f72da4')))
ensure('cumesh', 'cumesh',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/CuMesh.git', 'CuMesh',
                                  '12289e1062f0603f2f0d0771b02e1395d247f26f')))
ensure('flex_gemm', 'flex_gemm',
       lambda: _wheel(_clone('https://github.com/JeffreyXiang/FlexGEMM.git', 'FlexGEMM',
                                  '6dd94a859c26ee8246888502eada3dd8ad85532e')))
ensure('o_voxel', 'o_voxel',
       lambda: _wheel('/content/TRELLIS.2/o-voxel'))

print('\n✓ all CUDA extensions ready')

In [ ]:
#@title 7. Sanity check
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /content/TRELLIS.2
import o_voxel
from pillow_heif import register_heif_opener
register_heif_opener()
from trellis2.pipelines import Trellis2ImageTo3DPipeline  # prints the sparse backends line
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
print('✓ TRELLIS.2 imports OK (including HEIC/HEIF image support)')

## Demo 1 — Seven views without the straight-on front

Provide exactly these seven views of the same object, ideally spaced about 45° apart:

`front-left → left → back-left → back → back-right → right → front-right`

Do **not** include the straight-on front image in this run; it is reserved for Demo 2.
No camera poses are needed. Use the same object state, framing, and lighting throughout.
For an easy visual audit, prefix filenames `01_` through `07_` in the order above.
PNG, JPG/JPEG, WebP, HEIC, and HEIF images are accepted directly.

In [ ]:
#@title 8. Upload or select the seven non-front views
MULTIVIEW_IMAGE_SOURCE = 'upload'  #@param ["upload", "drive_folder"]
MULTIVIEW_DRIVE_FOLDER = '/content/drive/MyDrive/TRELLIS2_inputs/seven_views'  #@param {type:"string"}

import os, shutil
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp', '.heic', '.heif')
MULTIVIEW_INPUT_DIR = '/content/input_views/seven_views'
shutil.rmtree(MULTIVIEW_INPUT_DIR, ignore_errors=True)
os.makedirs(MULTIVIEW_INPUT_DIR)

if MULTIVIEW_IMAGE_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith(SUPPORTED_EXTENSIONS):
            with open(os.path.join(MULTIVIEW_INPUT_DIR, os.path.basename(name)), 'wb') as f:
                f.write(data)
        else:
            print(f'Ignoring unsupported file: {name}')
else:
    if not os.path.isdir(MULTIVIEW_DRIVE_FOLDER):
        raise FileNotFoundError(f'Drive input folder does not exist: {MULTIVIEW_DRIVE_FOLDER}')
    for name in sorted(os.listdir(MULTIVIEW_DRIVE_FOLDER)):
        if name.lower().endswith(SUPPORTED_EXTENSIONS):
            shutil.copy(os.path.join(MULTIVIEW_DRIVE_FOLDER, name), MULTIVIEW_INPUT_DIR)

MULTIVIEW_IMAGE_PATHS = sorted(
    os.path.join(MULTIVIEW_INPUT_DIR, f) for f in os.listdir(MULTIVIEW_INPUT_DIR)
    if f.lower().endswith(SUPPORTED_EXTENSIONS)
)
if len(MULTIVIEW_IMAGE_PATHS) != 7:
    found = ', '.join(os.path.basename(p) for p in MULTIVIEW_IMAGE_PATHS) or '(none)'
    raise ValueError(
        f'Expected exactly 7 non-front views, found {len(MULTIVIEW_IMAGE_PATHS)}: {found}. '
        'Supply front-left, left, back-left, back, back-right, right, and front-right; keep '
        'the straight-on front image for cell 13. Fix the selection and re-run this cell.'
    )
print(f'✓ seven non-front views loaded from {MULTIVIEW_INPUT_DIR}')

from PIL import Image
for path in MULTIVIEW_IMAGE_PATHS:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as exc:
        raise ValueError(f'Unreadable image {os.path.basename(path)}: {exc}') from exc

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 7, figsize=(21, 3.5))
for ax, p in zip(axes, MULTIVIEW_IMAGE_PATHS):
    with Image.open(p) as image:
        ax.imshow(image.convert('RGBA'))
    ax.set_title(os.path.basename(p), fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
#@title 9. Load the TRELLIS.2-4B pipeline (~15 GB download on first run)
from trellis2.pipelines import Trellis2ImageTo3DPipeline
if 'pipeline' not in globals():
    pipeline = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B')
    pipeline.cuda()  # low-VRAM mode: submodels move to GPU only while in use
print('✓ pipeline ready')

In [ ]:
#@title 10. Generate the seven-view 3D asset
MODE = 'stochastic'  #@param ["stochastic", "multidiffusion"]
RESOLUTION = 'default'  #@param ["default", "512", "1024", "1024_cascade", "1536_cascade"]
SEED = 42  #@param {type:"integer"}
PREPROCESS = True  #@param {type:"boolean"}
MULTIVIEW_OUTPUT_NAME = 'trellis2_7view'  #@param {type:"string"}
# MODE:       'stochastic' cycles through views across denoising steps (fast);
#             'multidiffusion' averages all views at every step (slower, more stable).
# RESOLUTION: 'default' uses the model's config (1024_cascade). Use '512' if you hit OOM.
# PREPROCESS: automatic background removal + recentering. Disable only for clean-alpha inputs.

import os, torch
assert MULTIVIEW_OUTPUT_NAME and os.path.basename(MULTIVIEW_OUTPUT_NAME) == MULTIVIEW_OUTPUT_NAME, (
    'MULTIVIEW_OUTPUT_NAME must be a plain filename.'
)
from PIL import Image, ImageOps

multiview_images = []
for path in MULTIVIEW_IMAGE_PATHS:
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        mode = 'RGBA' if ('A' in image.getbands() or 'transparency' in image.info) else 'RGB'
        multiview_images.append(image.convert(mode).copy())
multiview_mesh = pipeline.run_multi_image(
    multiview_images,
    seed=SEED,
    mode=MODE,
    pipeline_type=None if RESOLUTION == 'default' else RESOLUTION,
    preprocess_image=PREPROCESS,
)[0]
multiview_mesh.simplify(16777216)  # nvdiffrast limit
torch.cuda.empty_cache()
print(
    f'✓ seven-view mesh generated: {multiview_mesh.vertices.shape[0]:,} vertices, '
    f'{multiview_mesh.faces.shape[0]:,} faces'
)

In [ ]:
#@title 11. Render the seven-view turntable video
import os, cv2, imageio, torch
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap

OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

# Cell 12 parks the mesh on CPU to free VRAM for Demo 2; rendering needs it back on the GPU.
if multiview_mesh.device.type != 'cuda':
    multiview_mesh = multiview_mesh.cuda()

HDRI_PATH = 'assets/hdri/forest.exr'
hdri = cv2.imread(HDRI_PATH, cv2.IMREAD_UNCHANGED)
if hdri is None:
    raise FileNotFoundError(
        f'Could not read {HDRI_PATH}. Re-run cell 7 so the working directory is '
        '/content/TRELLIS.2 and OpenEXR support is enabled.'
    )
envmap = EnvMap(torch.tensor(
    cv2.cvtColor(hdri, cv2.COLOR_BGR2RGB), dtype=torch.float32, device='cuda'
))
multiview_video = render_utils.make_pbr_vis_frames(
    render_utils.render_video(multiview_mesh, envmap=envmap)
)
multiview_video_path = f'{OUT_DIR}/{MULTIVIEW_OUTPUT_NAME}.mp4'
imageio.mimsave(multiview_video_path, multiview_video, fps=15)

from IPython.display import Video, display
display(Video(multiview_video_path, embed=True, width=512))

In [ ]:
#@title 12. Export the seven-view GLB (and copy results to Drive)
EXPORT_REMESH = False  #@param {type:"boolean"}
REMESH_BAND = 1.0  #@param {type:"number"}
REMESH_PROJECT = 0.9  #@param {type:"number"}
# EXPORT_REMESH=False uses O-Voxel's duplicate/non-manifold/component cleanup path.
# Turn it on only when you specifically want narrow-band dual-contouring remeshing.

import gc, os, shutil, torch
import o_voxel

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# to_glb runs CUDA kernels (nvdiffrast / cumesh / flex_gemm). The teardown at the end of
# this cell parks the mesh on CPU, so move it back when the cell is re-run.
if multiview_mesh.device.type != 'cuda':
    multiview_mesh = multiview_mesh.cuda()

glb = o_voxel.postprocess.to_glb(
    vertices          = multiview_mesh.vertices,
    faces             = multiview_mesh.faces,
    attr_volume       = multiview_mesh.attrs,
    coords            = multiview_mesh.coords,
    attr_layout       = multiview_mesh.layout,
    voxel_size        = multiview_mesh.voxel_size,
    aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target = 1000000,
    texture_size      = 4096,
    remesh            = EXPORT_REMESH,
    remesh_band       = REMESH_BAND,
    remesh_project    = REMESH_PROJECT,
    verbose           = True,
)
multiview_glb_path = f'{OUT_DIR}/{MULTIVIEW_OUTPUT_NAME}.glb'
glb.export(multiview_glb_path)
print('✓ exported', multiview_glb_path)

if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/TRELLIS2_outputs'
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy(multiview_glb_path, drive_out)
    if 'multiview_video_path' in globals() and os.path.exists(multiview_video_path):
        shutil.copy(multiview_video_path, drive_out)
    print('✓ copied results to', drive_out)

from google.colab import files as colab_files
colab_files.download(multiview_glb_path)

# The GLB is on disk now. Park the mesh on CPU and drop the render frames so Demo 2 starts
# with a clean GPU; re-running cell 11 or 12 moves the mesh back automatically.
multiview_mesh = multiview_mesh.cpu()
for _name in ('multiview_video', 'glb'):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

## Demo 2 — Single straight-on front image

Use the held-out front image of the same object. Keeping the seed, resolution, and
preprocessing settings matched to Demo 1 makes the outputs easier to compare.

In [ ]:
#@title 13. Upload or select the held-out front image
SINGLE_IMAGE_SOURCE = 'upload'  #@param ["upload", "drive_file"]
SINGLE_IMAGE_DRIVE_PATH = '/content/drive/MyDrive/TRELLIS2_inputs/front.png'  #@param {type:"string"}

import os, shutil
SUPPORTED_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.webp', '.heic', '.heif')
SINGLE_INPUT_DIR = '/content/input_views/single_front'
shutil.rmtree(SINGLE_INPUT_DIR, ignore_errors=True)
os.makedirs(SINGLE_INPUT_DIR)

if SINGLE_IMAGE_SOURCE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    valid_uploads = [
        (name, data) for name, data in uploaded.items()
        if name.lower().endswith(SUPPORTED_EXTENSIONS)
    ]
    if len(valid_uploads) != 1:
        found = ', '.join(name for name, _ in valid_uploads) or '(none)'
        raise ValueError(
            f'Expected exactly 1 supported image, found {len(valid_uploads)}: {found}. '
            'Upload only the straight-on front view and re-run this cell.'
        )
    name, data = valid_uploads[0]
    SINGLE_IMAGE_PATH = os.path.join(SINGLE_INPUT_DIR, os.path.basename(name))
    with open(SINGLE_IMAGE_PATH, 'wb') as f:
        f.write(data)
else:
    if not os.path.isfile(SINGLE_IMAGE_DRIVE_PATH):
        raise FileNotFoundError(f'Drive input image does not exist: {SINGLE_IMAGE_DRIVE_PATH}')
    if not SINGLE_IMAGE_DRIVE_PATH.lower().endswith(SUPPORTED_EXTENSIONS):
        raise ValueError(f'Unsupported image type: {SINGLE_IMAGE_DRIVE_PATH}')
    SINGLE_IMAGE_PATH = os.path.join(SINGLE_INPUT_DIR, os.path.basename(SINGLE_IMAGE_DRIVE_PATH))
    shutil.copy(SINGLE_IMAGE_DRIVE_PATH, SINGLE_IMAGE_PATH)

from PIL import Image
from IPython.display import display
try:
    with Image.open(SINGLE_IMAGE_PATH) as image:
        image.verify()
except Exception as exc:
    raise ValueError(f'Unreadable image {os.path.basename(SINGLE_IMAGE_PATH)}: {exc}') from exc
with Image.open(SINGLE_IMAGE_PATH) as image:
    single_preview = image.convert('RGBA')
print('✓ held-out front image loaded:', os.path.basename(SINGLE_IMAGE_PATH))
display(single_preview)

In [ ]:
#@title 14. Generate the single-image 3D asset
SINGLE_RESOLUTION = 'default'  #@param ["default", "512", "1024", "1024_cascade", "1536_cascade"]
SINGLE_SEED = 42  #@param {type:"integer"}
SINGLE_PREPROCESS = True  #@param {type:"boolean"}
SINGLE_OUTPUT_NAME = 'trellis2_single_front'  #@param {type:"string"}

import os, torch
from PIL import Image, ImageOps
assert SINGLE_OUTPUT_NAME and os.path.basename(SINGLE_OUTPUT_NAME) == SINGLE_OUTPUT_NAME, (
    'SINGLE_OUTPUT_NAME must be a plain filename.'
)

with Image.open(SINGLE_IMAGE_PATH) as image:
    image = ImageOps.exif_transpose(image)
    mode = 'RGBA' if ('A' in image.getbands() or 'transparency' in image.info) else 'RGB'
    single_image = image.convert(mode).copy()
single_mesh = pipeline.run(
    single_image,
    seed=SINGLE_SEED,
    pipeline_type=None if SINGLE_RESOLUTION == 'default' else SINGLE_RESOLUTION,
    preprocess_image=SINGLE_PREPROCESS,
)[0]
single_mesh.simplify(16777216)  # nvdiffrast limit
torch.cuda.empty_cache()
print(
    f'✓ single-image mesh generated: {single_mesh.vertices.shape[0]:,} vertices, '
    f'{single_mesh.faces.shape[0]:,} faces'
)

In [ ]:
#@title 15. Render the single-image turntable video
import os, cv2, imageio, torch
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
from IPython.display import Video, display

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)

if single_mesh.device.type != 'cuda':
    single_mesh = single_mesh.cuda()

if 'envmap' not in globals():  # Demo 2 can be rendered without having run cell 11.
    HDRI_PATH = 'assets/hdri/forest.exr'
    hdri = cv2.imread(HDRI_PATH, cv2.IMREAD_UNCHANGED)
    if hdri is None:
        raise FileNotFoundError(
            f'Could not read {HDRI_PATH}. Re-run cell 7 so the working directory is '
            '/content/TRELLIS.2 and OpenEXR support is enabled.'
        )
    envmap = EnvMap(torch.tensor(
        cv2.cvtColor(hdri, cv2.COLOR_BGR2RGB), dtype=torch.float32, device='cuda'
    ))

single_video = render_utils.make_pbr_vis_frames(
    render_utils.render_video(single_mesh, envmap=envmap)
)
single_video_path = f'{OUT_DIR}/{SINGLE_OUTPUT_NAME}.mp4'
imageio.mimsave(single_video_path, single_video, fps=15)
display(Video(single_video_path, embed=True, width=512))

In [ ]:
#@title 16. Export the single-image GLB (and copy results to Drive)
SINGLE_EXPORT_REMESH = False  #@param {type:"boolean"}
SINGLE_REMESH_BAND = 1.0  #@param {type:"number"}
SINGLE_REMESH_PROJECT = 0.9  #@param {type:"number"}

import gc, os, shutil, torch
import o_voxel

OUT_DIR = globals().get('OUT_DIR', '/content/outputs')
os.makedirs(OUT_DIR, exist_ok=True)

if single_mesh.device.type != 'cuda':
    single_mesh = single_mesh.cuda()

single_glb = o_voxel.postprocess.to_glb(
    vertices          = single_mesh.vertices,
    faces             = single_mesh.faces,
    attr_volume       = single_mesh.attrs,
    coords            = single_mesh.coords,
    attr_layout       = single_mesh.layout,
    voxel_size        = single_mesh.voxel_size,
    aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target = 1000000,
    texture_size      = 4096,
    remesh            = SINGLE_EXPORT_REMESH,
    remesh_band       = SINGLE_REMESH_BAND,
    remesh_project    = SINGLE_REMESH_PROJECT,
    verbose           = True,
)
single_glb_path = f'{OUT_DIR}/{SINGLE_OUTPUT_NAME}.glb'
single_glb.export(single_glb_path)
print('✓ exported', single_glb_path)

if USE_DRIVE:
    drive_out = '/content/drive/MyDrive/TRELLIS2_outputs'
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy(single_glb_path, drive_out)
    if 'single_video_path' in globals() and os.path.exists(single_video_path):
        shutil.copy(single_video_path, drive_out)
    print('✓ copied results to', drive_out)

from google.colab import files as colab_files
colab_files.download(single_glb_path)

single_mesh = single_mesh.cpu()
globals().pop('single_video', None)
gc.collect()
torch.cuda.empty_cache()

print('\nComparison ready:')
if 'multiview_glb_path' in globals():
    print('  seven-view :', multiview_glb_path)
print('  single     :', single_glb_path)

## Troubleshooting

| Symptom | Fix |
|---|---|
| Runtime verification fails | Choose **A100** and **Runtime Version 2025.10** under Runtime → Change runtime type, then reconnect. |
| `detected CUDA version … mismatches` during a build | A 12.5/12.6 warning is expected; a major-version error means the wrong Colab runtime is selected. |
| Import error / `undefined symbol` from a cached extension | Delete `Drive/TRELLIS2_cache/wheels/` and re-run cell 6. |
| `401/403` when downloading models | Accept the licenses for `facebook/dinov3-vitl16-pretrain-lvd1689m` and `briaai/RMBG-2.0` on Hugging Face, and check your `HF_TOKEN` Colab secret. |
| Seven-view upload is rejected | Supply exactly seven supported files: front-left, left, back-left, back, back-right, right, and front-right. Keep the straight-on front image for cell 13. |
| HEIC/HEIF image is unreadable | Rerun cell 4 to install `pillow-heif`, then cell 7 to register its Pillow decoder before uploading again. |
| CUDA OOM during generation | Set `RESOLUTION='512'` in cell 10 or `SINGLE_RESOLUTION='512'` in cell 14 and re-run that demo. |
| O-Voxel fails with `Eigen/Dense` or `Eigen/Core` not found | Rerun cell 6; it installs `libeigen3-dev` and links the headers into `o-voxel/third_party/eigen/`. |
| `Could not read assets/hdri/forest.exr` | Re-run cell 7 — the render cells read the HDRI relative to `/content/TRELLIS.2`. |
| Build fails on `git clone` of a pinned repo | Transient network error; just re-run cell 6. Successful wheels are already cached. |
| FlashAttention wheel is rejected | Confirm cell 1 reports Python 3.12, PyTorch 2.8, and runtime 2025.10; the notebook uses the matching official wheel. |
| A GLB has disconnected shards | Keep `EXPORT_REMESH=False` in cell 12 or `SINGLE_EXPORT_REMESH=False` in cell 16 so O-Voxel runs its full topology-cleanup path. |

**Re-running on the same session**: rerun cells 8→12 for a new seven-view set, or cells
13→16 for a new single front image. The pipeline stays loaded. Cells 12 and 16 park their
mesh on CPU when they finish so the other demo gets a clean GPU; re-running the matching
render or export cell moves it back automatically, so you only need to re-run generation
when you change an input or a generation parameter.